# SpherIDiff with Per-Degree Channel Mixing

This notebook tests a new spectral stage for the pooled SpherIDiff. In the earlier models, the spectral stage applied one dense weight over the spherical harmonic coefficients, shared by all channels. It mixed the coefficients with each other but never mixed the channels, and it held 73% of the parameters of the pooled model. A model with a low-rank weight reached almost the same validation error, which suggests that most of these parameters were not used well.

The new stage keeps the coefficients separate and mixes the channels instead, with a matrix that depends on the degree:

$$\hat g_{\ell m} = \hat f_{\ell m}\, A(\ell) + \frac{m}{L}\, \hat f_{\ell,-m}\, B(\ell), \qquad A(\ell) = \sum_k \phi_k(\ell)\, W^A_k, \quad B(\ell) = \sum_k \phi_k(\ell)\, W^B_k .$$

- **Isotropic part.** $A(\ell)$ mixes the channels with a matrix that changes with the degree, so each scale can relate the variables in its own way.
- **East-west part.** $B(\ell)$ acts on the longitude derivative of the field, which in real harmonics swaps the cosine and sine coefficients of each order.
- **Knots.** Both matrices are interpolated from 8 knots in the degree with hat functions, and the coarsest level has one knot per degree (7).

The dense weight and the coefficient filters are removed. The rest of the block is unchanged: a spherical convolution at the end of the two blocks of the finest level, the channel MLP, the temporal attention and the noise conditioning.

| Model | Spectral stage | Parameters |
|---|---|---|
| `pooled3_disco` (earlier result) | dense mode mixing with coefficient filters | 2,958,206 |
| `pooled3_channel` (this notebook) | per-degree channel mixing | 2,415,897 |

The variant is pre-trained on the global ERA5 data at 5.6 degrees for 25,000 steps (`pilot_25k`) and evaluated under the fixed protocol `final_v1` on the test year 2021. It is then fine-tuned for 5,000 steps (`finetune`) on the regional patch at 1.5 degrees over South Asia, with the Slepian levels L' = 120, 60 and 30, and evaluated on that patch zero-shot and after fine-tuning. On the patch, the operator of each spectral stage is restricted to the Slepian basis of its level. The pre-trained knots are kept frozen, new knots cover the degrees above the pre-trained band limit and start at zero, and a trainable update in the Slepian basis is added, which also starts at zero. The pointwise layers, the attention and the spherical convolution are fine-tuned.

The notebook is designed for a background run on a T4 GPU. It reads the global data cache from the dataset `thesyeddaniyal/era5-pilot-6degrees` and the regional data cache from the dataset `thesyeddaniyal/era5-1-5-southasia`. The pre-trained weights are created in this session. The results are packed into `channel_mixing_results.zip` in the working folder.

In [ ]:
import os
import sys
from pathlib import Path

import torch

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
RUNS_ROOT = WORK_ROOT / "runs"
FINAL_PROTOCOL = "final_v1"
ZERO_SHOT_SUFFIX = "_zeroshot"
VARIANTS = ["pooled3_channel"]
PRETRAINING = ["pilot_" + name for name in VARIANTS]
FINETUNING = ["finetune_" + name for name in VARIANTS]
ZERO_SHOT = [name + ZERO_SHOT_SUFFIX for name in FINETUNING]
ARCHIVE = WORK_ROOT / "channel_mixing_results.zip"


## Inputs and hardware

The repository code is cloned from GitHub (`https://github.com/syeddaniyalg/spheri-diff`) into the working folder, which requires internet access for the notebook. Within each attached dataset, the data cache is the folder that holds `stats.npz`. When the expected input path of a dataset does not exist, the whole of `/kaggle/input` is searched for a folder of the expected name.

In [ ]:
import subprocess

REPOSITORY_URL = "https://github.com/syeddaniyalg/spheri-diff"
CODE_ROOT = WORK_ROOT / "spheri-diff"
GLOBAL_INPUT = INPUT_ROOT / "datasets" / "thesyeddaniyal" / "era5-pilot-6degrees"
REGIONAL_INPUT = INPUT_ROOT / "datasets" / "thesyeddaniyal" / "era5-1-5-southasia"


def locate_cache(expected, folder_name):
    roots = [expected] if expected.exists() else [INPUT_ROOT]
    for root in roots:
        for current, _, files in os.walk(root):
            if "stats.npz" in files and (expected.exists() or Path(current).name == folder_name):
                return Path(current)
    raise FileNotFoundError(f"no data cache named {folder_name} was found under {roots[0]}")


if not CODE_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(CODE_ROOT)], check=True)
GLOBAL_CACHE = locate_cache(GLOBAL_INPUT, "era5_5p6deg")
REGIONAL_CACHE = locate_cache(REGIONAL_INPUT, "era5_1p5deg_south_asia")
sys.path.insert(0, str(CODE_ROOT / "src"))
print("code:", CODE_ROOT)
print("global data:", GLOBAL_CACHE)
print("regional data:", REGIONAL_CACHE)
for index in range(torch.cuda.device_count()):
    print("gpu", index, torch.cuda.get_device_name(index))

## Running the jobs

The variant runs as one queue of separate processes on the first GPU, with five jobs in order: pre-training, the global final evaluation, fine-tuning, the regional final evaluation of the zero-shot model and the regional final evaluation of the fine-tuned model. Each job needs the output of the job before it, so a second GPU would stay idle.

Every job prints its lines into the notebook, prefixed with the elapsed time, the GPU and the job name, and the same lines are kept in `runs/_logs/parallel.log`. The jobs and the packing of the results share one cell with a `try` and a `finally`, so the figures, the status file, the summary tables and the archive are produced even when a job fails or the cell is interrupted. The archive is also refreshed every 30 minutes.

In [ ]:
from spheridiff.analysis.final_table import write_final_summary
from spheridiff.kaggle.finalise import finalise, guarded, refresh_archive
from spheridiff.kaggle.jobs import final_job, finetune_job, place_queues, train_job
from spheridiff.kaggle.runner import ParallelRunner, RunContext

GPUS = torch.cuda.device_count()
CONTEXT = RunContext(CODE_ROOT, RUNS_ROOT, RUNS_ROOT / "_logs", GLOBAL_CACHE)
REGIONAL_SUMMARY = RUNS_ROOT / "regional"


def variant_jobs(name):
    pretraining, finetuning = "pilot_" + name, "finetune_" + name
    return [
        train_job(pretraining, GLOBAL_CACHE),
        final_job(pretraining, FINAL_PROTOCOL, cache_root=GLOBAL_CACHE),
        finetune_job(finetuning, RUNS_ROOT, REGIONAL_CACHE),
        final_job(finetuning, FINAL_PROTOCOL, finetuning + ZERO_SHOT_SUFFIX, REGIONAL_CACHE),
        final_job(finetuning, FINAL_PROTOCOL, cache_root=REGIONAL_CACHE),
    ]


def queue(variants):
    return [job for name in variants for job in variant_jobs(name)]


def regional_summary():
    REGIONAL_SUMMARY.mkdir(parents=True, exist_ok=True)
    write_final_summary(REGIONAL_SUMMARY, [(RUNS_ROOT, ZERO_SHOT + FINETUNING)])


PLACEMENT = [(0, queue(VARIANTS))]


def refresh():
    return refresh_archive(RUNS_ROOT, ARCHIVE)


RUNNER = None
try:
    RUNNER = ParallelRunner(place_queues(PLACEMENT, GPUS), CONTEXT, refresh=refresh)
    RUNNER.run()
finally:
    OUTCOMES = RUNNER.outcomes() if RUNNER else []
    RUNS_ROOT.mkdir(parents=True, exist_ok=True)
    guarded(RUNS_ROOT, "regional_summary", regional_summary)
    finalise(RUNS_ROOT, ARCHIVE, OUTCOMES, PRETRAINING, final_names=PRETRAINING)
print(OUTCOMES)
print(ARCHIVE, f"{ARCHIVE.stat().st_size / 1024**2:.1f} MB")

## Results

The pre-training and fine-tuning summaries list the steps, training time, time per step, peak memory and final loss. The figures show the validation error during pre-training. The global table (`final_evaluation_summary.txt`) lists the pre-trained model under the fixed protocol on the global test year. The regional table (`regional/final_evaluation_summary.txt`) lists the model on the patch, zero-shot (0 trained steps) and after fine-tuning. Both tables give the number of trainable parameters, the trained steps, the forecast scores, and for each task and method the normalised error on the unobserved entries, with the reference of the task as the last row. The earlier result for comparison is `pooled3_disco`: a validation error of 0.305 after pre-training and 0.324 on the patch after fine-tuning.

In [ ]:
import json

for name in PRETRAINING + FINETUNING:
    summary = RUNS_ROOT / name / "summary.json"
    print(name, json.loads(summary.read_text()) if summary.exists() else "no summary, see runs/_logs")

for table in [RUNS_ROOT / "final_evaluation_summary.txt", REGIONAL_SUMMARY / "final_evaluation_summary.txt"]:
    if table.exists():
        print(table.read_text(encoding="utf-8"))

In [ ]:
from IPython.display import Image, display

for figure in ["evaluation_curves.png", "training_loss.png"]:
    path = RUNS_ROOT / figure
    if path.exists():
        display(Image(filename=str(path)))